# CampusPulse — Findings, Conclusions & Next Steps
### `04_conclusions.ipynb` · What the analytics revealed and what to do next

---

This notebook translates every chart from `03_analytics.ipynb` into **specific, numbered findings** you can directly use in your project report and viva.  

It is structured as:
1. Finding-by-finding interpretation (one finding = one observation from the data)  
2. What each finding means for the next pipeline step  
3. A consolidated conclusions table  
4. The recommended next steps notebook structure


---
## 1 · Findings from Correlation Analysis

### Finding C1 — Study hours is the dominant signal in the Lifestyle dataset

From **Image 7** (Lifestyle correlation heatmap):

| Relationship | r value | Strength |
|---|---|---|
| study_hours ↔ stress_encoded | **+0.74** | Strong positive |
| study_hours ↔ GPA | **+0.73** | Strong positive |
| stress_encoded ↔ GPA | +0.55 | Moderate positive |
| physical_activity ↔ stress | −0.20 | Weak negative |
| sleep ↔ stress | −0.30 | Weak-moderate negative |

**Interpretation:**  
Study hours drives both higher GPA *and* higher stress simultaneously. This is not contradictory — it reflects the real-world trade-off where academically ambitious students work more, achieve more, and are also under more pressure. The dataset captures both effects together. This means **GPA and stress are not opposites** — they co-move with study hours as the common driver.

**Note on synthetic data:** Because this is an algorithmically generated dataset, the r=0.74 for study_hours↔stress is likely *built into the generation model*. Treat these specific values as patterns to explore, not ground truth.

---

### Finding C2 — Physical activity has a broad negative correlation pattern

**physical_activity_hours_per_day** has moderate negative correlations with *every other feature*:
- vs study_hours: r = −0.49  
- vs sleep_hours: r = −0.47  
- vs extracurricular: r = −0.37  
- vs GPA: r = −0.34  

**Interpretation:**  
This is a **time-budget effect**, not a causal finding. Students have a fixed 24 hours per day. Students who spend more time on physical activity necessarily have less time for everything else — including study and sleep. The negative correlations are a mathematical artifact of the time-budget constraint, *not* evidence that exercise harms GPA.

**Viva-critical:** If asked "why does physical activity correlate negatively with GPA?", the correct answer is: *compositional constraint — daily hours sum to 24, so any one activity being high forces the others to be lower on average.*

---

### Finding C3 — UCI grade multicollinearity: G1, G2, G3 are near-redundant

From **Image 16** (UCI correlation zoom):

| Pair | r |
|---|---|
| G1 ↔ G2 | 0.86 |
| G2 ↔ G3 | 0.91 |
| G1 ↔ G3 | 0.81 |

**Interpretation:**  
Knowing G1 gives you 81% of the information about G3. Including all three in a predictive model would triple-count the same academic signal and inflate the apparent importance of "grade history". For the dashboard's ML components, **use only G3** as the target/feature. For exploratory analysis, all three can be retained to study grade trajectory.

**Direct consequence for next step:** When you build the stress predictor or clustering for UCI data, drop G1 and G2 from the feature set.

---

### Finding C4 — extracurricular and social hours are near-zero correlated with stress

From **Image 7**:  
- extracurricular ↔ stress: r = −0.01 (essentially zero)  
- social_hours ↔ stress: r = −0.06 (essentially zero)  

**Interpretation:**  
These two features carry **no linear information about stress level**. This will be confirmed by the mutual information analysis. These features are candidates for **dropping before supervised ML** — keeping useless features adds noise to models without improving accuracy.


---
## 2 · Findings from Distribution Analysis

### Finding D1 — Study hours KDE shows near-perfect stress group separation

From **Image 8** (left KDE plot):

The three stress group KDE curves for study hours are **almost completely non-overlapping**:
- Low stress: peak ~5.5 hrs/day (narrow, tight distribution)
- Moderate stress: peak ~7.5 hrs/day (bimodal)
- High stress: peak ~8.5–9.5 hrs/day (wide, spread out)

**Interpretation:**  
This is the clearest single-feature discriminator in the entire dataset. If you know a student's study hours, you can predict their stress group with high accuracy from this feature alone. This explains the r=0.74 in C1 but goes further — the KDE shows the separation is **clean**, not just correlated.

**Practical implication:** In the stress predictor (next notebook), study_hours alone as a single feature will give a strong baseline. Any model that achieves much better than this baseline is using other features meaningfully.

---

### Finding D2 — GPA KDE shows partial but meaningful separation

From **Image 8** (right KDE plot):

- Low stress: GPA peaks around 2.75–2.85
- Moderate stress: GPA peaks around 3.0–3.1  
- High stress: GPA peaks around 3.2–3.4 (rightward shift — **higher GPA**)

**Interpretation:**  
High-stress students actually have *higher* GPA on average. This is the same paradox as C1 — stress and academic performance are positively correlated here because both are driven by high study hours. The three curves overlap considerably, meaning GPA alone is a weaker stress predictor than study hours.

---

### Finding D3 — Physical activity shrinks at High stress (violin plot)

From **Image 15** (Physical Activity violin):

The violin for High stress is **narrow at the bottom and concentrated around 2–6 hrs**, while Low stress has a wide violin spanning up to 12–13 hrs.

**Interpretation:**  
High-stress students are squeezing out physical activity — consistent with the time-budget interpretation from C2. The wide Low-stress violin at high physical activity values suggests that students who maintain exercise have the time budget to do so, which correlates with having lower academic/study pressure.

---

### Finding D4 — Extracurricular and social hours show flat, overlapping violins across stress groups

From **Image 15** (Extracurricular and Social panels):

Both show nearly **identical violin shapes** across Low, Moderate, and High stress.

**Interpretation:**  
Confirms C4 — these features do not discriminate stress levels. Social and extracurricular time appears to be maintained regardless of stress level, suggesting students protect social time even under academic pressure. This is an interesting sociological finding worth mentioning in your report.


---
## 3 · Findings from Outlier Detection

### Finding O1 — The dataset is remarkably clean; only physical_activity and GPA have outliers

From **Image 11**:
- physical_activity_hours_per_day: **5 outliers** (students at 12–13 hrs/day — extreme athletes)
- GPA: **4 outliers** (students near 4.0 GPA — maximum achievers)
- All other features: **0 outliers**

**Interpretation:**  
The 0-outlier count for study, sleep, extracurricular, and social hours is a strong signal that this is **synthetic/generated data** — real survey data almost always has messy outliers. In a real dataset you'd expect 1–5% outlier rates in every column. This is useful to state in your report's data quality section.

**Decision:** Keep all 9 outliers — they are at the extreme ends of plausible values (not impossible) and the count is too small to affect any analysis with n=2000.

---

### Finding O2 — UCI has genuine outliers in grades (G3 = 0)

Not shown in Image 11 but identified in the cleaning notebook: **~65 students with G3=0** (across both subjects combined). These were flagged as `possible_dropout` rather than removed.

**Interpretation:**  
Including dropouts biases the distribution downward. For grade prediction models, always filter to `possible_dropout == 0`. For dropout analysis, these rows are precisely the target.


---
## 4 · Findings from PCA

### Finding P1 — Lifestyle: no dominant single factor; 3 PCs needed for 80% variance

From **Image 13** (Scree plot):

| PCs kept | Cumulative variance |
|---|---|
| PC1 only | 35.8% |
| PC1 + PC2 | 58.5% |
| PC1–PC3 | **79.5% ≈ 80%** |
| PC1–PC4 | 95.3% |

**Interpretation:**  
The variance is spread relatively evenly across components — there is no single factor that explains the majority of the data (no PC above 40%). This is consistent with the correlation heatmap showing no single feature dominating all others. The data represents genuinely multi-dimensional variation in student lifestyles.

**For t-SNE and clustering:** Use all 6 features (or the top 4 PCs covering 95%) as input. Don't reduce to just PC1 and PC2 before clustering — you'd lose 40% of the variance.

---

### Finding P2 — PCA biplot: study_hours and GPA load together on PC1; physical_activity loads oppositely

From **Image 12** (Biplot):

- **study hours** and **GPA** arrows point to the upper-right (+PC1, +PC2)  
- **physical activity** arrow points to the upper-left (−PC1, +PC2)  
- **sleep, social, extracurricular** arrows cluster downward (−PC2)  

**Interpretation:**  
PC1 is essentially an **"academic intensity" axis**: students on the right are high-study, high-GPA, high-stress. Students on the left are high-physical-activity, lower-study. PC2 roughly separates students who sleep/socialise more (bottom) from those who exercise more (top).

The biplot also visually shows what C1 found mathematically — study hours and GPA occupy the same direction in feature space, confirming they co-vary strongly.

---

### Finding P3 — PCA biplot: stress groups separate primarily along PC1

From **Image 12** (point colors):

- Green (Low stress) points concentrate in the **left half** of the plot  
- Red (High stress) points concentrate in the **right half**  
- The separation is along PC1 — the academic intensity axis  

**Interpretation:**  
PC1 is not just a mathematical construct — it has a **semantic interpretation as academic pressure**. Students on the right of PC1 are high-study, high-GPA, and predominantly High stress. This is the most important structural finding from PCA: academic intensity (not any single feature) is the primary organising axis of this population.

---

### Finding P4 — UCI PCA: 8 components needed for 80% variance (much more complex than Lifestyle)

From **Image 2** (UCI scree plot):

PC1 only explains ~25% of UCI variance vs 35.8% for Lifestyle. Reaching 80% requires ~8 components for UCI.

**Interpretation:**  
UCI is a **richer, more complex dataset** — 33 original features capturing family background, school support, social behaviour, alcohol use, and grades. None of these dominates. The UCI data has more independent axes of variation because it measures genuinely independent life dimensions, not just time allocation.

**Consequence:** For UCI clustering, more features are useful and dimensionality reduction is more important to prevent the curse of dimensionality.

---

### Finding P5 — UCI PCA 2D: low performers (Q1, red) visibly separate in the left region

From **Image 2** (UCI 2D scatter):

Q1 (low-grade) students cluster distinctly in the **left portion** of the PC1 axis (negative PC1 values). Q4 students spread more to the right.

**Interpretation:**  
Even with only 25.3% + 14.4% = ~40% variance explained, the 2D projection already separates academic performance groups. The remaining 60% variance (behavioral, family, social features) adds noise but the grade signal is strong enough to be visible in 2D PCA. A classifier should do well on UCI grade prediction.


---
## 5 · Findings from t-SNE

### Finding T1 — High stress forms a spatially distinct region in t-SNE space

From **Image 14** (left panel — colored by stress):

High stress (pink/red) points occupy a **distinct right-side region** of the t-SNE embedding. Low stress (green) concentrates on the left. Moderate stress sits in between with significant overlap with both groups.

**Interpretation:**  
The natural neighbourhood structure of the data already **separates High stress students from Low stress students** without any supervision. This means the features collectively carry enough structure for a classifier to distinguish stress levels — the signal is not just in one feature (study hours) but in the combination.

---

### Finding T2 — The SAME spatial structure appears when colored by GPA quartile

From **Image 14** (right panel — colored by GPA quartile):

- Q1 (low GPA, pink) maps almost exactly to the same region as Low stress  
- Q4 (high GPA, green) maps almost exactly to High stress region  

**Interpretation:**  
This is the **strongest visual confirmation** of the study_hours↔GPA↔stress triangle:  
Low GPA ≈ Low stress ≈ Low study hours (left region)  
High GPA ≈ High stress ≈ High study hours (right region)  

These three variables form a single coherent behavioral pattern — not three separate independent phenomena. A student who studies a lot will tend to have a higher GPA *and* higher stress simultaneously. The t-SNE makes this visible in a single image.

---

### Finding T3 — t-SNE reveals moderate stress as genuinely intermediate (not a distinct group)

The Moderate stress group (yellow/orange) does **not** form a tight separate cluster — it bleeds into both Low and High regions throughout the embedding.

**Interpretation:**  
For classification purposes, Moderate vs High stress is a **harder boundary** than Low vs High. A binary classifier (Low vs High) will perform better than a 3-class classifier. This is worth testing explicitly in the ML notebook.


---
## 6 · Findings from Mutual Information

### Finding M1 — study_hours carries dramatically more information than all other features combined

From **Image 10** (left panel):

| Feature | MI Score | Signal level |
|---|---|---|
| study_hours_per_day | **0.65** | Dominant |
| gpa | ~0.18 | Moderate |
| sleep_hours_per_day | ~0.14 | Moderate |
| physical_activity_hours_per_day | ~0.02 | Below threshold |
| extracurricular_hours_per_day | ~0.01 | Below threshold |
| social_hours_per_day | ~0.01 | Below threshold |

**Interpretation:**  
The MI score of 0.65 for study_hours is more than **3× the next feature** (GPA at ~0.18). This is a dominant signal. The three lowest features (physical activity, extracurricular, social) all fall below the 0.05 threshold — they provide near-zero additional information about stress after study hours is known.

---

### Finding M2 — Rank disagreement on physical_activity reveals non-linearity

From **Image 10** (comparison of MI vs Pearson):

- **Pearson ranking**: physical_activity ranked 4th (|r|=0.20)  
- **MI ranking**: physical_activity ranked 4th–5th (MI≈0.02, below threshold)  

Both methods agree this feature is weak, but the Pearson |r|=0.20 suggests *some* linear relationship while MI says near-zero *any* relationship. This means the r=0.20 from Pearson is **a weak signal that doesn't hold up under a more sensitive test**. The non-linear test says: knowing physical activity hours tells you essentially nothing about stress.

**Consequence for feature selection:** Build the stress predictor with only: `study_hours`, `gpa`, `sleep_hours`. Drop physical_activity, extracurricular, social. This 3-feature model will be simpler, more interpretable, and likely more generalisable.


---
## 7 · Findings from K-Means Clustering

### Finding K1 — Elbow (K=3) and silhouette (K=2) disagree — use K=3 for interpretability

From **Image 9**:

- Elbow method: K=3 (clear bend in WCSS curve)
- Silhouette score: highest at K=2 (score ≈ 0.222)
- Silhouette drops monotonically from K=2 to K=6, then slightly recovers

**Interpretation:**  
The silhouette saying K=2 reflects that the data has **two broad super-clusters** (high-intensity students vs low-intensity students). But the elbow saying K=3 captures a finer split that aligns with the three stress levels (Low/Moderate/High). 

For a **dashboard and report**, K=3 is more useful and interpretable — it maps naturally to the Low/Moderate/High stress labels already in the data. The fact that silhouette prefers K=2 is an honest limitation to state: the cluster boundaries are not crisp.

---

### Finding K2 — Cluster 0 is "Academic Overloaders"; Cluster 1 is "Balanced Students"

From **Images 5 and 6** (K=2 was run, but interpret as the two dominant groups):

**Cluster 0** (n ≈ 960):
- study_hours mean = **8.53** hrs/day
- sleep_hours mean = 7.79 hrs  
- physical_activity = **2.30** hrs (low)
- gpa mean = **3.32**
- Stress composition: **77% High stress, 23% Moderate, 0% Low**

**Cluster 1** (n ≈ 1040):
- study_hours mean = **6.38** hrs/day
- sleep_hours mean = 7.22 hrs  
- physical_activity = **5.77** hrs (high — 2.5× Cluster 0)
- gpa mean = **2.92**
- Stress composition: 27% High, 44% Moderate, **29% Low**

**Interpretation:**  
The two clusters are clean and meaningful:
- **Cluster 0 = Academic Overloaders**: Study hard, achieve high GPA, sacrifice physical activity, almost universally High stress. The "grind" student profile.  
- **Cluster 1 = Balanced Students**: Study moderately, exercise substantially, spread across all stress levels. The "healthy balance" student profile.

This is the headline finding for your dashboard — every student can be assigned to one of these two profiles with a simple clustering model.

---

### Finding K3 — Cluster membership strongly predicts stress, but is not perfectly deterministic

From **Image 5** (contingency heatmap):

Cluster 0 → 77% High stress. Cluster 1 → only 27% High stress.

But Cluster 1 still has 27% High-stress students. This means **cluster membership is a strong but not perfect stress predictor**. There are balanced students who are still stressed — likely due to factors not captured in the lifestyle hours (exam pressure, personal circumstances).

**Implication for dashboard:** Show cluster profile as a "student type" card, not a definitive stress prediction. The supervised classifier (next step) will be more accurate than the cluster label alone.


---
## 8 · Findings from Cross-Dataset Comparison

### Finding X1 — Lifestyle and UCI show opposite stress-performance relationships

From **Image 3** (cross-dataset box plots):

**Lifestyle dataset:**
- Low stress → median academic percentile: **17th**
- Moderate stress → **39th**
- High stress → **69th**
- *Pattern: Higher stress → better academic performance*

**UCI dataset:**
- Low stress (proxy) → median: **56th**
- Moderate stress → **29th**
- High stress → **19th**
- *Pattern: Higher stress → worse academic performance* (opposite!)

**Interpretation:**  
This is the most analytically important cross-dataset finding. The two datasets show **reversed patterns** for stress vs academic performance. This is not a data error — it reflects a fundamental difference:

- In the **Lifestyle dataset**, stress is *driven by high study hours* (the generation model connects them directly). Students who study more get stressed AND get better grades. Stress here is a byproduct of effort.

- In the **UCI dataset**, the stress proxy is built from alcohol use, failures, and low freetime. Students with academic *failures* and substance use have higher stress — *and* lower grades. Stress here is a byproduct of struggle.

**These are two completely different mechanisms giving opposite correlations.** This is precisely why you cannot merge datasets blindly — the "stress vs performance" direction would change sign depending on which dataset dominates the merge.

---

### Finding X2 — Mental Health dataset shows no stress-performance gradient (flat medians)

From **Image 3** (right panel):

All three stress groups in the Mental Health dataset have median academic percentile around **31st–54th** with heavily overlapping interquartile ranges. There is no clear gradient.

**Interpretation:**  
Three possible explanations:
1. The small sample size (n=101) means there isn't enough power to detect the gradient even if it exists.
2. The CGPA range string measurement is imprecise — the midpoint approximation loses information.
3. In this population, mental health status (depression/anxiety) does not strongly co-vary with academic performance in the same way as study hours or failure history do in the other datasets.

**For your report:** This is an honest null result — state it clearly. The mental health dataset is most valuable for the mental health prevalence statistics (what % have depression/anxiety), not for academic performance analysis.

---

### Finding X3 — Mental Health CGPA is bimodal; other datasets are unimodal

From **Image 4** (KDE of academic percentile):

- Lifestyle and UCI: roughly uniform/normal distributions (as expected from percentile transformation)
- Mental Health: **strong bimodal distribution** with peaks at ~25th and ~75th percentile

**Interpretation:**  
The bimodality in the Mental Health dataset is a genuine data artifact from the **CGPA range encoding**. The original CGPA range "3.00–3.49" gets mapped to 3.245, and "3.50–4.00" gets mapped to 3.75. Since most students fall into two or three discrete CGPA bands, the midpoint transformation creates discrete piles — which appear as peaks in the KDE after percentile ranking.

**Implication:** When using CGPA from the Mental Health dataset, prefer **ordinal analysis** (which CGPA band is a student in?) over continuous analysis (what is the exact GPA?). The continuous approximation introduces artificial peaks.


---
## 9 · Consolidated Findings Table

| # | Finding | Source plot | Impact on next step |
|---|---------|-------------|---------------------|
| C1 | study_hours dominates with r=0.74 to stress and GPA simultaneously | Heatmap | Use study_hours as primary feature in all models |
| C2 | Physical activity's negative correlations are time-budget artifacts | Heatmap | Do not interpret as "exercise harms performance"; use with caution in models |
| C3 | G1/G2/G3 multicollinearity r=0.86–0.91 | UCI heatmap | Drop G1 and G2 from all UCI ML features |
| C4 | extracurricular and social near-zero correlated with stress | Heatmap | Drop both from stress predictor feature set |
| D1 | Study hours KDE: 3 stress groups almost perfectly separated | KDE plot | study_hours single-feature baseline will be strong |
| D2 | GPA KDE: high-stress students have higher GPA (paradox) | KDE plot | Report the paradox; explain the mechanism |
| D3 | Physical activity shrinks at high stress (violin) | Violin | Confirms time-budget; do not use as independent predictor |
| D4 | Extracurricular and social: flat across stress groups | Violin | Confirms dropping from feature set |
| O1 | 0 outliers in most columns (confirms synthetic data) | Boxplots | State this in data quality section of report |
| P1 | 3 PCs needed for 80% Lifestyle variance | Scree | Use full 6 features for ML, not PCA-reduced |
| P2 | PC1 = "academic intensity" axis (study+GPA vs physical_activity) | Biplot | Semantic label for PC1 in report |
| P3 | Stress groups separate along PC1 | Biplot | Confirms PC1 is the most discriminative axis |
| P4 | UCI needs 8 PCs for 80% variance | UCI scree | UCI is genuinely higher-dimensional; feature selection important |
| P5 | UCI low performers visibly separate in 2D PCA | UCI scatter | Grade prediction should work well in UCI |
| T1 | High stress is spatially distinct in t-SNE | t-SNE | Unsupervised structure supports supervised learning |
| T2 | t-SNE colors match for stress AND GPA quartile | t-SNE | Single coherent behavioral pattern confirmed |
| T3 | Moderate stress is not a distinct cluster | t-SNE | Binary classifier (Low vs High) may outperform 3-class |
| M1 | study_hours MI=0.65, all others ≪ 0.20 | MI chart | Final feature set: study_hours, gpa, sleep_hours |
| M2 | physical_activity: Pearson r=0.20 but MI≈0.02 | MI vs Pearson | Non-linear relationship likely; drop from stress model |
| K1 | Elbow K=3, silhouette K=2 — use K=3 for interpretability | Elbow/Silhouette | Run K=3 for final clustering to match stress labels |
| K2 | Cluster 0 = Academic Overloaders; Cluster 1 = Balanced | Cluster profiles | Use as "student type" label in dashboard |
| K3 | Cluster 0 → 77% High stress but not 100% | Contingency | Cluster is a signal, not a definitive diagnosis |
| X1 | Lifestyle: High stress → better grades; UCI: High stress → worse grades | Cross-dataset boxes | **Opposite mechanisms — never merge these datasets** |
| X2 | Mental Health: no stress-performance gradient | Cross-dataset boxes | Use MH dataset for prevalence stats, not performance analysis |
| X3 | Mental Health CGPA is bimodal (range-string artifact) | Cross-dataset KDE | Use ordinal CGPA bands for MH, not continuous midpoint |


---
## 10 · What These Findings Mean for the Next Notebooks

### 10.1 · Feature Set Decisions (apply from now on)

```python
# LIFESTYLE — features to use in all ML models
LS_ML_FEATURES = [
    "study_hours_per_day",        # MI=0.65, r=0.74 — dominant
    "gpa",                        # MI=0.18, r=0.55 — moderate
    "sleep_hours_per_day",        # MI=0.14, r=0.30 — moderate
    # DROPPED: extracurricular (MI≈0.01), social (MI≈0.01), physical_activity (MI≈0.02)
]
LS_TARGET = "stress_level"   # 3-class: Low / Moderate / High
LS_BINARY_TARGET = ...       # map Low=0, (Moderate+High)=1  OR  (Low+Moderate)=0, High=1

# UCI — features to use in ML models
UCI_ML_FEATURES = [
    "studytime", "failures", "absences",
    "dalc", "walc", "freetime", "goout",
    # G3 only — not G1 or G2 (multicollinear)
]
UCI_TARGET = "g3"            # regression OR binned into Pass/Fail
# Filter: df_uc[df_uc["possible_dropout"] == 0]

# MENTAL HEALTH — use for prevalence analysis, not performance prediction
MH_ANALYSIS_COLS = ["depression", "anxiety", "panic_attack", "cgpa_range"]
```

---

### 10.2 · Next Notebook — `05_ml_models.ipynb`

Build three distinct supervised models:

**Model 1: Stress Classifier on Lifestyle data**
- Input: `study_hours`, `gpa`, `sleep_hours`
- Target: `stress_level` (3-class)  
- Models to try: Random Forest, Logistic Regression, SVM
- Baseline to beat: predict the majority class (always "High") → check what accuracy that gives
- Expected finding from our analysis: RF should achieve >85% accuracy because KDE shows near-perfect separation
- Also try binary (Low vs High only) per Finding T3 — expect higher accuracy

**Model 2: Grade Predictor on UCI data**
- Input: `studytime`, `failures`, `absences`, `dalc`, `walc`, `freetime`
- Target: `g3` as regression (MAE, R²) OR binned as Pass (G3≥10) / Fail (G3<10)
- Expected from P5: grade separation visible in PCA → linear model may already work well

**Model 3: Mental Health Risk Flag (exploratory)**
- Input: `cgpa_midpoint`, `year_of_study`
- Target: `depression` or composite stress_score
- n=101 — treat results as exploratory only, always caveat

---

### 10.3 · Dashboard Architecture (from these findings)

| Dashboard page | Data source | What to show | Key finding used |
|---|---|---|---|
| **Student Type Classifier** | Lifestyle (clustered) | "You are: Academic Overloader / Balanced Student" card | K2 |
| **Stress Predictor** | Lifestyle ML model | Input study/sleep/GPA → predict stress level | M1, D1 |
| **Feature Importance** | Lifestyle MI + Pearson | Side-by-side bar chart with disagreements highlighted | M2 |
| **Grade Predictor** | UCI ML model | Input student profile → predict G3 | P5 |
| **Mental Health Prevalence** | Mental Health | % with depression, anxiety, panic; by year of study | X2 |
| **Cross-Dataset Comparison** | Standardized table | Stress vs academic percentile — show the opposite patterns | X1 |
| **PCA Explorer** | Lifestyle / UCI | Interactive biplot — hover to see student details | P2, P3 |

---

### 10.4 · One-paragraph viva answer: "What did your analysis find?"

> *"Our analysis identified study hours per day as the dominant predictor of both stress and academic performance in the Lifestyle dataset, with a Pearson r of 0.74 and mutual information score of 0.65 — more than three times the next feature. PCA confirmed this semantically: PC1 represents an 'academic intensity' axis that explains 35.8% of lifestyle variance, and stress groups separate clearly along this axis. However, the most important cross-dataset finding is that the direction of this relationship reverses across populations: in the Lifestyle dataset, higher stress correlates with better academic performance because both are driven by study hours; in the UCI dataset, higher stress — proxied from failures and alcohol use — correlates with worse grades. This demonstrates why the three datasets cannot be merged and must be analysed separately. The clustering analysis produced two interpretable student profiles: 'Academic Overloaders' who study 8.5 hrs/day, achieve higher GPA, but are 77% High-stress; and 'Balanced Students' who study 6.4 hrs/day, exercise more, and show a healthier stress distribution."*
